# Deep Speech — Scaling up end-to-end speech recognition (2014)

_Papers · Speech & Audio_

**Throw away the phoneme dictionary, the pronunciation model, and the HMM aligner: turn a spectrogram straight into letters with one recurrent network trained by CTC.**

---

This notebook is a practice scaffold for the **Deep Speech — Scaling up end-to-end speech recognition (2014)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import numpy as np, torch, torch.nn as nn

# ============================================================
# (1) WORKED CTC EXAMPLE — must match the lesson's 'example'
#     alphabet {blank=0, h=1, i=2}, T=4 frames, target "hi"
# ============================================================
y = torch.tensor([[0.1, 0.7, 0.2],
                  [0.6, 0.3, 0.1],
                  [0.2, 0.2, 0.6],
                  [0.5, 0.1, 0.4]])          # softmax outputs, rows sum to 1
log_probs = y.log().unsqueeze(1)             # shape (T=4, N=1, C=3)
ctc = nn.CTCLoss(blank=0, reduction='none')
loss = ctc(log_probs, torch.tensor([[1, 2]]),       # target "hi"
           torch.tensor([4]), torch.tensor([2]))    # input_len=4, target_len=2
print("worked CTC loss:", round(loss.item(), 6),
      " hand 0.738145  allclose:",
      torch.allclose(loss, torch.tensor([0.738145]), atol=1e-5))
# greedy best-path decode of this example -> "hi"
sym = {0:'_', 1:'h', 2:'i'}
def collapse(path):
    out, prev = [], None
    for c in path:
        if c != prev: out.append(c); prev = c
    return ''.join(sym[c] for c in out if c != 0)
print("greedy decode:", collapse(y.argmax(1).tolist()))   # -> hi

# ============================================================
# (2) TINY SPECTROGRAM -> RNN -> CTC PIPELINE on toy audio-like data
# ============================================================
F, T = 8, 12
VOCAB = ['_', 'h', 'i', 'a', 't']            # index 0 = blank (CTC requirement)

def make_spectrogram(target, seed):
    """toy 'audio': noise + a 2.0 energy band per character, in time order."""
    rng = np.random.RandomState(seed)
    spec = rng.randn(T, F) * 0.3             # (T frames, F freq bins)
    seg = T // len(target)
    for k, ch in enumerate(target):
        f = VOCAB.index(ch) % F              # each char -> a frequency band
        spec[k*seg:(k+1)*seg, f] += 1.5
    return spec.astype(np.float32)

data  = [("hi",1),("hat",2),("it",3),("ha",4),("hit",5),("ait",6),("ti",7),("aht",8)]
specs = [torch.tensor(make_spectrogram(t, s)) for t, s in data]
targs = [[VOCAB.index(c) for c in t] for t, _ in data]

class DeepSpeechTiny(nn.Module):
    def __init__(self, bidir=True):
        super().__init__()
        H = 16
        self.fc1 = nn.Linear(F, H); self.fc2 = nn.Linear(H, H); self.fc3 = nn.Linear(H, H)
        self.rnn = nn.RNN(H, H, bidirectional=bidir)          # the one recurrent layer (layer 4)
        self.out = nn.Linear(H * (2 if bidir else 1), len(VOCAB))
        self.g = lambda z: z.clamp(0, 20)    # clipped ReLU  g(z)=min{max{0,z},20}
    def forward(self, x):                    # x: (T, F)
        h = self.g(self.fc1(x)); h = self.g(self.fc2(h)); h = self.g(self.fc3(h))
        h, _ = self.rnn(h.unsqueeze(1))      # (T, 1, H*dir)
        return self.out(h).log_softmax(-1)   # (T, 1, V)  -> log-probs for CTC

def greedy(lp):
    am, out, prev = lp.squeeze(1).argmax(-1).tolist(), [], None
    for c in am:
        if c != prev: out.append(c); prev = c
    return ''.join(VOCAB[c] for c in out if c != 0)

def train(bidir, epochs=80):
    torch.manual_seed(0)
    net = DeepSpeechTiny(bidir)
    opt = torch.optim.Adam(net.parameters(), lr=0.02)
    ctc = nn.CTCLoss(blank=0)
    for _ in range(epochs):
        opt.zero_grad(); total = 0
        for spec, tg in zip(specs, targs):
            lp = net(spec)
            total = total + ctc(lp, torch.tensor([tg]),
                                torch.tensor([T]), torch.tensor([len(tg)]))
        total.backward(); opt.step()
    correct = sum(greedy(net(spec)) == t for (t, _), spec in zip(data, specs))
    return round(total.item(), 3), correct, [greedy(net(spec)) for spec in specs]

lb, cb, db = train(bidir=True)
lu, cu, du = train(bidir=False)               # ABLATION: forward-only RNN
print(f"BIDIR  : final CTC loss {lb}  decoded {cb}/8  -> {db}")
print(f"UNIDIR : final CTC loss {lu}  decoded {cu}/8  -> {du}")  # ablation: much worse

## Visualize the data & results

_Deep Speech's one recurrent layer is bidirectional. If we ablate it to a same-size unidirectional (forward-only) RNN and train for the same 80 epochs, how much does end-to-end character decoding suffer?_

In [ ]:
import numpy as np, torch, torch.nn as nn

F, T = 8, 12
VOCAB = ['_', 'h', 'i', 'a', 't']            # blank = 0

def make_spectrogram(target, seed):
    rng = np.random.RandomState(seed)
    spec = rng.randn(T, F) * 0.3
    seg = T // len(target)
    for k, ch in enumerate(target):
        spec[k*seg:(k+1)*seg, VOCAB.index(ch) % F] += 1.5
    return spec.astype(np.float32)

data  = [("hi",1),("hat",2),("it",3),("ha",4),("hit",5),("ait",6),("ti",7),("aht",8)]
specs = [torch.tensor(make_spectrogram(t, s)) for t, s in data]
targs = [[VOCAB.index(c) for c in t] for t, _ in data]

class Net(nn.Module):
    def __init__(self, bidir=True):
        super().__init__(); H = 16
        self.fc1, self.fc2, self.fc3 = nn.Linear(F,H), nn.Linear(H,H), nn.Linear(H,H)
        self.rnn = nn.RNN(H, H, bidirectional=bidir)
        self.out = nn.Linear(H*(2 if bidir else 1), len(VOCAB))
        self.g = lambda z: z.clamp(0, 20)
    def forward(self, x):
        h = self.g(self.fc1(x)); h = self.g(self.fc2(h)); h = self.g(self.fc3(h))
        h, _ = self.rnn(h.unsqueeze(1)); return self.out(h).log_softmax(-1)

def greedy(lp):
    am, out, prev = lp.squeeze(1).argmax(-1).tolist(), [], None
    for c in am:
        if c != prev: out.append(c); prev = c
    return ''.join(VOCAB[c] for c in out if c != 0)

def train(bidir, epochs=80):
    torch.manual_seed(0)
    net = Net(bidir); opt = torch.optim.Adam(net.parameters(), lr=0.02); ctc = nn.CTCLoss(blank=0)
    for _ in range(epochs):
        opt.zero_grad(); total = 0
        for spec, tg in zip(specs, targs):
            total = total + ctc(net(spec), torch.tensor([tg]),
                                torch.tensor([T]), torch.tensor([len(tg)]))
        total.backward(); opt.step()
    correct = sum(greedy(net(spec)) == t for (t, _), spec in zip(data, specs))
    return round(total.item(), 3), correct

lb, cb = train(True); lu, cu = train(False)
print("Bidirectional :", lb, "loss,", cb, "/8 decoded")   # 0.097, 8/8
print("Unidirectional:", lu, "loss,", cu, "/8 decoded")   # 2.446, 1/8

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Greedy-decode the per-frame argmax path a a _ b _ b (where _ is blank) under CTC's collapse rule. What text comes out?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- First merge adjacent repeated symbols. — _CTC collapses runs of the same emitted label to one: 'a a' → 'a', and the two 'b's are separated by a blank so they are NOT adjacent._
- Then delete every blank. — _Blank means 'emit nothing here'; it only serves to separate repeats and to fill empty frames._

**Answer:** After merging repeats: a _ b _ b. After dropping blanks: "abb". The blank between the two b's is what keeps them as two separate letters — without a blank, 'b b' would collapse to a single 'b'.

</details>

**Problem 2.** In the worked example (alphabet {blank,h,i}, T=4), the loss for target "hi" was $-\ln 0.478 = 0.738$. If instead every frame put probability 1.0 on the single best path h _ i _, what would the CTC loss be, and why is the real loss larger?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the loss if all probability sits on one collapsing path. — _Then $\mathbb{P}(\text{hi}\mid x)=1$, so the loss is $-\ln 1 = 0$._
- Compare to the real 0.478 total. — _Real probability is spread across 15 alignments AND across wrong strings, so $\mathbb{P}(\text{hi})\lt 1$, making $-\ln \mathbb{P}\gt 0$._

**Answer:** It would be $-\ln 1 = 0$ (perfect). The real loss is 0.738 because the softmax also assigns probability to frames/paths that do NOT collapse to "hi" (and spreads the rest across 15 valid alignments), so $\mathbb{P}(\text{hi}\mid x)=0.478\lt 1$. Training pushes this toward 0 by concentrating probability on hi-producing paths.

</details>

**Problem 3.** ABLATION. Deep Speech's layer 4 is bidirectional. Predict and then explain: if you replace it with a same-size unidirectional (forward-only) RNN, same training budget, what happens to the toy decode accuracy — and why would future context help decide a letter at the current frame?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run CODE/CODEVIZ with bidirectional=True then bidirectional=False at the SAME 80 epochs. — _Holds everything fixed except direction, isolating the effect of seeing future sound._
- Read off final CTC loss and how many of 8 toy utterances decode correctly. — _Lower loss + more exact-match decodes = the directional information actually helped._
- Reason about acoustics. — _A sound's identity often depends on what FOLLOWS it (coarticulation); a forward-only model must commit before hearing the rest._

**Answer:** In our run the bidirectional model reaches CTC loss 0.097 and decodes 8/8 toy utterances; the unidirectional ablation at the same 80 epochs is stuck at loss 2.446 and decodes only 1/8 (our small run, not the paper's numbers). Future context matters because a frame's correct letter can depend on later frames — the bidirectional layer lets each output see the whole utterance, which is exactly why Deep Speech uses one.

</details>